# Transformers — The Architecture That Changed Everything

Transformers (Vaswani et al., 2017 — "Attention Is All You Need") are THE architecture
behind GPT, BERT, Claude, Llama, and every modern language model. Understanding transformers
is the single most important thing in modern AI.

This notebook builds the intuition from scratch, then implements self-attention in PyTorch.

---
## Why Transformers?

Before transformers, the dominant architecture was **RNNs** (Recurrent Neural Networks):

```
RNN: word₁ → h₁ → word₂ → h₂ → word₃ → h₃ → ... → wordₙ → hₙ
     ← must process sequentially, one word at a time →
```

**Problems with RNNs:**

| Problem | Explanation |
|---|---|
| **Sequential** | Can't parallelize — word 5 must wait for words 1-4 |
| **Vanishing gradients** | By word 100, the model has "forgotten" word 1 |
| **Slow training** | Sequential = no GPU parallelism |

**Transformers solve all three:**

```
Transformer: [word₁, word₂, word₃, ..., wordₙ] → processed ALL AT ONCE in parallel
             Every word can directly attend to every other word
```

The secret sauce? **Self-Attention**.

---
## Self-Attention — The Key Mechanism

For each word in a sentence, self-attention asks: **"How much should I pay attention to every other word?"**

```
"The cat sat on the mat because it was tired"

For the word "it":
  → high attention to "cat"    (what "it" refers to)
  → some attention to "tired"  (describing "it")
  → low attention to "the"     (not informative)
```

This is how transformers handle context — "bank" in "river bank" will attend to "river",
while "bank" in "bank account" will attend to "account". Same word, different context,
different representation.

The result: **context-dependent embeddings** that change based on surrounding words.

---
## Attention Score Computation — Q, K, V

Self-attention uses three matrices: **Query (Q)**, **Key (K)**, and **Value (V)**.

Analogy: searching a library.
- **Query** = your question ("What's related to 'cat'?")
- **Key** = each book's label (what each word advertises about itself)
- **Value** = each book's content (the actual information to retrieve)

Steps:
1. Project each word embedding into Q, K, V using learned weight matrices
2. Compute attention scores: how well does each Query match each Key?
3. Scale and softmax → attention weights (sum to 1)
4. Weighted sum of Values → new context-aware representation

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

The $\sqrt{d_k}$ prevents dot products from getting too large (which would make softmax
too peaked, crushing gradients).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

sentence = ["The", "cat", "sat", "on", "the", "mat"]
seq_len = len(sentence)
d_model = 4  # tiny embedding dimension for clarity

X = torch.randn(seq_len, d_model)
print(f"Input embeddings shape: {X.shape}  ({seq_len} words × {d_model} dims)")
print(f"\nEach row is a word embedding:")
for word, vec in zip(sentence, X):
    print(f"  {word:>5}: {vec.numpy().round(2)}")

In [ ]:
d_k = d_model  # dimension of Q, K, V (typically d_model // num_heads)

W_q = torch.randn(d_model, d_k)
W_k = torch.randn(d_model, d_k)
W_v = torch.randn(d_model, d_k)

Q = X @ W_q  # (6, 4) × (4, 4) → (6, 4)
K = X @ W_k
V = X @ W_v

print(f"Q shape: {Q.shape}  (each word has a query vector)")
print(f"K shape: {K.shape}  (each word has a key vector)")
print(f"V shape: {V.shape}  (each word has a value vector)")

In [ ]:
scores = Q @ K.T  # (6, 4) × (4, 6) → (6, 6)
print(f"Raw attention scores: {scores.shape}")
print(f"scores[i][j] = how much word i attends to word j\n")

scores_scaled = scores / (d_k ** 0.5)
print(f"After scaling by √d_k = √{d_k} = {d_k**0.5:.1f}:")
print(scores_scaled.detach().numpy().round(2))

In [ ]:
attention_weights = F.softmax(scores_scaled, dim=-1)

print("Attention weights (each row sums to 1):")
print(attention_weights.detach().numpy().round(3))
print(f"\nRow sums: {attention_weights.sum(dim=-1).detach().numpy().round(3)}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(attention_weights.detach().numpy(), cmap='Blues')
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(sentence, fontsize=12)
ax.set_yticklabels(sentence, fontsize=12)
ax.set_xlabel('Attending TO', fontsize=12)
ax.set_ylabel('Attending FROM', fontsize=12)
ax.set_title('Self-Attention Weights', fontsize=14)

for i in range(seq_len):
    for j in range(seq_len):
        val = attention_weights[i, j].item()
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9)

plt.colorbar(im)
plt.tight_layout()
plt.show()

print("Read as: row word attends to column word with that weight.")
print("In a trained model, meaningful patterns emerge (pronouns attend to their referents, etc).")

In [ ]:
output = attention_weights @ V  # (6, 6) × (6, 4) → (6, 4)

print(f"Output shape: {output.shape}  (same as input — each word now has a context-aware vector)\n")
print("Before attention (original embeddings):")
for word, vec in zip(sentence, X):
    print(f"  {word:>5}: {vec.detach().numpy().round(2)}")

print("\nAfter attention (context-aware):")
for word, vec in zip(sentence, output):
    print(f"  {word:>5}: {vec.detach().numpy().round(2)}")

print("\nEach word's vector now incorporates information from all other words.")

---
## Multi-Head Attention

One attention head captures one type of relationship. But language has many:
- **Head 1** might learn syntactic relationships (subject-verb)
- **Head 2** might learn coreference ("it" → "cat")
- **Head 3** might learn semantic similarity

**Multi-head attention** runs several attention mechanisms in parallel, each with its
own Q, K, V projections, then concatenates the results.

```
MultiHead(Q, K, V) = Concat(head₁, head₂, ..., headₕ) × W_O
where headᵢ = Attention(Q × Wᵢᑫ, K × Wᵢᴷ, V × Wᵢⱽ)
```

If `d_model = 512` and `num_heads = 8`, each head works in `d_k = 64` dimensions.
The total computation is the same as single-head attention, just split up.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        scores = (Q @ K.transpose(-2, -1)) / (self.d_k ** 0.5)
        weights = F.softmax(scores, dim=-1)
        
        attended = (weights @ V).transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        return self.W_o(attended), weights

mha = MultiHeadAttention(d_model=32, num_heads=4)
x = torch.randn(2, 6, 32)  # batch=2, seq_len=6, d_model=32
output, weights = mha(x)

print(f"Input:  {x.shape}  (2 sentences × 6 words × 32 dims)")
print(f"Output: {output.shape}  (same shape — context-enriched)")
print(f"Attention weights: {weights.shape}  (2 batches × 4 heads × 6 × 6)")

---
## Positional Encoding

Attention is permutation-invariant: it doesn't know if a word is at position 1 or 100.
But word order matters ("dog bites man" ≠ "man bites dog").

**Positional encoding** injects order information by adding a position-dependent vector
to each word embedding.

The original paper used sinusoidal functions:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Different frequencies at different dimensions → unique pattern for each position.

In [ ]:
def positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model))
    
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

pe = positional_encoding(50, 64)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(pe.numpy(), aspect='auto', cmap='RdBu')
axes[0].set_xlabel('Embedding Dimension')
axes[0].set_ylabel('Position')
axes[0].set_title('Positional Encoding Matrix')
plt.colorbar(im, ax=axes[0])

for dim in [0, 1, 4, 5, 20, 21]:
    axes[1].plot(pe[:, dim].numpy(), label=f'dim {dim}')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Value')
axes[1].set_title('Positional Encoding by Dimension')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Low dimensions vary rapidly (fine position info), high dimensions vary slowly (coarse info).")
print("Each position gets a unique fingerprint.")

---
## The Transformer Block

A single transformer block stacks several components:

```
Input Embeddings + Positional Encoding
        │
        ▼
┌─────────────────────────┐
│   Multi-Head Attention  │
└────────────┬────────────┘
        │ + residual connection
        ▼
┌─────────────────────────┐
│     Layer Norm          │
└────────────┬────────────┘
        │
        ▼
┌─────────────────────────┐
│   Feed-Forward Network  │  (2 linear layers with ReLU/GELU)
└────────────┬────────────┘
        │ + residual connection
        ▼
┌─────────────────────────┐
│     Layer Norm          │
└────────────┬────────────┘
        │
        ▼
      Output
```

**Residual connections** (add the input back to the output) prevent vanishing gradients.
**Layer norm** stabilizes training.

Real models stack 12–96 of these blocks.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        attended, weights = self.attention(x)
        x = self.norm1(x + self.dropout(attended))  # residual + norm
        x = self.norm2(x + self.dropout(self.ff(x)))  # residual + norm
        return x, weights

block = TransformerBlock(d_model=64, num_heads=4, d_ff=256)
x = torch.randn(2, 10, 64)  # batch=2, seq_len=10, d_model=64
out, attn = block(x)

print(f"Input:  {x.shape}")
print(f"Output: {out.shape}  (same shape — transformers preserve dimensions)")
print(f"\nTotal parameters: {sum(p.numel() for p in block.parameters()):,}")

---
## Encoder vs Decoder

The original transformer has both an encoder and decoder. Modern models often use just one:

| Architecture | How It Works | Best For | Examples |
|---|---|---|---|
| **Encoder-only** | Sees all tokens at once (bidirectional) | Understanding: classification, NER, similarity | BERT, RoBERTa, DistilBERT |
| **Decoder-only** | Sees only past tokens (causal/left-to-right) | Generation: text, code, chat | GPT, Llama, Claude |
| **Encoder-Decoder** | Encoder processes input, decoder generates output | Transformation: translation, summarization | T5, BART, mBART |

The key difference is the **attention mask**:
- Encoder: every token can see every other token
- Decoder: each token can only see tokens BEFORE it (causal masking)

In [ ]:
seq_len = 5

encoder_mask = torch.ones(seq_len, seq_len)
decoder_mask = torch.tril(torch.ones(seq_len, seq_len))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
labels = [f'tok{i+1}' for i in range(seq_len)]

for ax, mask, title in [(axes[0], encoder_mask, 'Encoder (Bidirectional)'),
                         (axes[1], decoder_mask, 'Decoder (Causal)')]:
    ax.imshow(mask, cmap='Greens', vmin=0, vmax=1)
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Can see')
    ax.set_ylabel('Token')
    for i in range(seq_len):
        for j in range(seq_len):
            val = '✓' if mask[i, j] == 1 else '✗'
            ax.text(j, i, val, ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.show()

print("Encoder: tok3 sees ALL tokens → great for understanding the full sentence")
print("Decoder: tok3 sees only tok1, tok2, tok3 → needed for generating the NEXT token")

---
## BERT — Bidirectional Encoder Representations from Transformers

BERT (Google, 2018) uses only the **encoder** and pre-trains with two tasks:

**1. Masked Language Model (MLM):** randomly mask 15% of tokens and predict them.
```
Input:  "The cat [MASK] on the mat"
Target: "sat"
```
This forces BERT to understand context from BOTH directions.

**2. Next Sentence Prediction (NSP):** given two sentences, is B the actual next sentence?
```
A: "The cat sat on the mat."  B: "It purred happily."     → IsNext
A: "The cat sat on the mat."  B: "Stock prices rose."     → NotNext
```

After pre-training on massive data, BERT can be **fine-tuned** for downstream tasks
(classification, NER, QA) by adding a small head on top.

| Model | Layers | Hidden | Heads | Parameters |
|---|---|---|---|---|
| BERT-base | 12 | 768 | 12 | 110M |
| BERT-large | 24 | 1024 | 16 | 340M |

---
## GPT — Generative Pre-trained Transformer

GPT (OpenAI) uses only the **decoder** and pre-trains with one task:

**Autoregressive language modeling:** predict the next token.
```
Input:  "The cat sat on the"
Target: "mat"
```

Each token can only attend to tokens BEFORE it (causal masking).

At generation time, it produces text one token at a time:
```
"The" → "cat" → "sat" → "on" → "the" → "mat" → "."
```

| Model | Layers | Parameters | Context Length |
|---|---|---|---|
| GPT-2 | 48 | 1.5B | 1024 tokens |
| GPT-3 | 96 | 175B | 2048 tokens |
| GPT-4 | ~120? | ~1.8T (rumored) | 128K tokens |

The scaling trend: more parameters + more data = emergent capabilities.
GPT-3 could do few-shot learning. GPT-4 can reason, code, and pass bar exams.

---
## Implement Self-Attention from Scratch

Let's build a complete, clean implementation and verify it works.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights

torch.manual_seed(0)
seq_len, d_k = 4, 8
Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_k)

output_bi, weights_bi = scaled_dot_product_attention(Q, K, V)

causal_mask = torch.tril(torch.ones(seq_len, seq_len))
output_causal, weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print("Bidirectional attention weights:")
print(weights_bi.numpy().round(3))
print("\nCausal attention weights:")
print(weights_causal.numpy().round(3))
print("\nNotice: in causal, upper triangle is 0 — future tokens are invisible.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
tokens = ['w1', 'w2', 'w3', 'w4']

for ax, w, title in [(axes[0], weights_bi, 'Bidirectional (BERT-style)'),
                      (axes[1], weights_causal, 'Causal (GPT-style)')]:
    im = ax.imshow(w.detach().numpy(), cmap='Purples', vmin=0)
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(tokens)
    ax.set_yticklabels(tokens)
    ax.set_title(title, fontsize=12)
    for i in range(seq_len):
        for j in range(seq_len):
            ax.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center', fontsize=10)

plt.tight_layout()
plt.show()

---
## Putting It All Together — A Mini Transformer

In [ ]:
class MiniTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = positional_encoding(max_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        self.output_head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        seq_len = x.shape[1]
        x = self.embedding(x) + self.pos_encoding[:seq_len].unsqueeze(0)
        
        all_weights = []
        for block in self.blocks:
            x, weights = block(x)
            all_weights.append(weights)
        
        logits = self.output_head(x)
        return logits, all_weights

mini = MiniTransformer(
    vocab_size=1000,
    d_model=64,
    num_heads=4,
    d_ff=256,
    num_layers=3
)

tokens = torch.randint(0, 1000, (2, 20))  # batch of 2, seq_len 20
logits, attn_weights = mini(tokens)

total_params = sum(p.numel() for p in mini.parameters())

print(f"Input:  {tokens.shape}  (2 sequences × 20 tokens)")
print(f"Output: {logits.shape}  (2 sequences × 20 positions × 1000 vocab)")
print(f"\nTotal parameters: {total_params:,}")
print(f"Attention weights per layer: {attn_weights[0].shape}")
print(f"\nFor reference:")
print(f"  BERT-base:  110,000,000 params")
print(f"  GPT-2:    1,500,000,000 params")
print(f"  GPT-3:  175,000,000,000 params")
print(f"  Our mini:   {total_params:>13,} params")

---
## The Transformer Zoo — Scale is All You Need

In [ ]:
import pandas as pd

models_data = {
    'Model': ['BERT-base', 'BERT-large', 'GPT-2', 'GPT-3', 'GPT-4 (est.)', 'Llama-2 7B', 
              'Llama-2 70B', 'T5-base', 'T5-large'],
    'Type': ['Encoder', 'Encoder', 'Decoder', 'Decoder', 'Decoder', 'Decoder',
             'Decoder', 'Enc-Dec', 'Enc-Dec'],
    'Parameters': ['110M', '340M', '1.5B', '175B', '~1.8T', '7B',
                   '70B', '220M', '770M'],
    'Best For': ['Classification, NER, QA', 'Same (higher quality)', 
                 'Text generation', 'Few-shot, reasoning', 'Everything',
                 'Open-source generation', 'High-quality open generation',
                 'Translation, summarization', 'Same (higher quality)'],
}

df = pd.DataFrame(models_data)
print(df.to_string(index=False))

---
## Key Concepts Summary

The attention formula — memorize this:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

**What each piece does:**

| Component | Purpose |
|---|---|
| Self-Attention | Let each token look at all other tokens |
| Multi-Head | Multiple attention patterns in parallel |
| Positional Encoding | Inject word order information |
| Residual Connections | Prevent vanishing gradients |
| Layer Norm | Stabilize training |
| Feed-Forward Network | Non-linear transformation per position |

**The evolution:**
```
BoW/TF-IDF → Word2Vec/GloVe → RNNs/LSTMs → Transformers → LLMs
(no meaning)   (static meaning)  (sequential)  (parallel + context)  (scale)
```

**Next notebook:** Using these models in practice with Hugging Face — the sklearn of NLP.